Imports

In [103]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

Get Soup Function

In [104]:
def get_soup(url):
    page = requests.get(url)
    soup = BeautifulSoup(page.content, "html.parser")
    
    return soup

Get Category of the book from Book Page Function

In [105]:
def get_category(book):
    book_url = book.find("h3").find("a").get("href")
    book_url =f"https://books.toscrape.com/catalogue/{book_url}"
    soup = get_soup(book_url)

    book_category = soup.find("ul",class_="breadcrumb").find_all("li")[2].get_text(strip=True)

    return book_category



Get books from the page Function

In [106]:
def get_books_from_page(i):
    # Get Page
    url = f"https://books.toscrape.com/catalogue/page-{i}.html" 
    soup = get_soup(url)

    books=soup.find_all("article",class_="product_pod")
    return books

Get Book Data Function

In [107]:
def get_book_data(book, rating_map):

    # Book Name
    book_name = book.find("a",title =True).get("title")
    
    # Rating
    rating_text = book.find("p", class_="star-rating")["class"][1]
    book_rating = rating_map[rating_text]
    
    # Get Book Price and Availability
    book_info = book.find("div", class_="product_price")

    # Get Book Price
    book_price = book_info.find("p", class_="price_color").get_text(strip= True)
    book_price = float(book_price.replace("£", ""))

    # Get Book Availability
    is_available = book_info.find("p", class_="instock availability").text.strip()

    # Get Category
    book_category = get_category(book)
            
    return{"Book Name" :book_name,
        "Price" : book_price,
        "Rating" : book_rating,
        "Availability" : is_available,
        "Category" : book_category
        }  
            

# Main Code

In [ ]:
data = []

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

for i in range(1, 51):

    books = get_books_from_page(i)

    for book in books:

        book_data = get_book_data(
            book,
            rating_map
        )

        data.append(book_data)

books_df = pd.DataFrame(data)
books_df